# ProxyLens 01: QLoRA fine-tune of Qwen2.5-3B (resolution extraction)

**What:** fine-tunes `Qwen2.5-3B-Instruct` with QLoRA (4-bit, LoRA r=16) to turn one AGM/EGM
agenda item into `ResolutionExtraction` JSON, distilled from the teacher's labels.

**Inputs:** the private HF dataset `<you>/proxylens-sft-v1` (from `scripts/push_sft_to_hub.py`);
Colab secret `HF_TOKEN` with write access.

**Outputs:** LoRA adapter at `<you>/proxylens-qwen3b-lora-v1` and a `Q4_K_M` GGUF at
`<you>/proxylens-qwen3b-gguf-v1` (both private), plus `metrics.json` with val JSON validity and
type accuracy.

**Runtime:** free T4, about 2 h per epoch (~1,370 examples, ~1,500 tokens each; ~80 s per
optimizer step). Default is 1 epoch: a 2-epoch run gave no gain on the val subset. Checkpoints go to Google Drive every 10 steps; after a
disconnect, re-run all cells and training resumes from the last checkpoint.

**Hyper-parameters (SPEC 7.3):** r=16, alpha=16, dropout 0, all attention + MLP projections;
lr 2e-4 cosine, 1 epoch (see the config cell), effective batch 16 (1 x 16 to fit T4 memory), fp16 on T4,
seed 3407; loss on assistant tokens only.

## 1. Install and configure

In [ ]:
%%capture
# Unsloth pulls matching torch / transformers / trl / peft / bitsandbytes for Colab.
!pip install -q unsloth
!pip install -q "huggingface_hub>=0.25" "pydantic>=2.8"

In [ ]:
import os
# Must be set before torch starts: reduces fragmentation-driven OOMs on the 15 GB T4.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from unsloth import FastLanguageModel  # import before transformers/trl: Unsloth patches them

import glob, json, time
import torch
from google.colab import drive, userdata
from huggingface_hub import HfApi, snapshot_download

print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

HF_TOKEN = userdata.get("HF_TOKEN")  # Colab secret; needs write access
HF_USER = HfApi(token=HF_TOKEN).whoami()["name"]
DATASET_REPO = f"{HF_USER}/proxylens-sft-v1"           # pushed by scripts/push_sft_to_hub.py
LORA_REPO = f"{HF_USER}/proxylens-qwen3b-lora-v1"
GGUF_REPO = f"{HF_USER}/proxylens-qwen3b-gguf-v1"
GGUF_FILE = "proxylens-q4_k_m.gguf"                     # = STUDENT_GGUF_FILE in .env
BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
MAX_SEQ_LEN = 4096        # longest SFT example is ~4.1k tokens
MAX_NEW_TOKENS = 1024     # longest teacher answer is ~870 tokens
SEED = 3407
print("HF user:", HF_USER)

import transformers
transformers.utils.logging.set_verbosity_error()  # hides a harmless per-generate max_length warning

In [ ]:
# First run (v1): 2 epochs gave val loss 0.191 -> 0.182 and identical subset metrics
# (96.9% valid JSON, 93.8% type accuracy) after epochs 1 and 2, so 1 epoch is the default.
EPOCHS = 1
FULL_VAL_EVAL = False     # generating all 137 val answers adds ~1 h; notebook 02 / Phase 7 evaluate properly
LR = 2e-4
# Effective batch 16 as in SPEC 7.3. One sequence per device step: two ~4k-token sequences
# plus the 152k-vocab logits do not fit in a T4's 15 GB.
BATCH, GRAD_ACCUM = 1, 16
EVAL_SUBSET = 32          # val items generated after each epoch (full val runs at the end)
drive.mount("/content/drive")
OUTPUT_DIR = "/content/drive/MyDrive/proxylens/qwen3b-lora-v1"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 2. Data

In [ ]:
from datasets import load_dataset

data_dir = snapshot_download(DATASET_REPO, repo_type="dataset", token=HF_TOKEN,
                             allow_patterns=["sft/*"])
ds = load_dataset("json", data_files={"train": f"{data_dir}/sft/train.jsonl",
                                      "val": f"{data_dir}/sft/val.jsonl"})
print(ds)
print(ds["train"][0]["messages"][1]["content"][:400])

In [ ]:
# Copied from app/schemas/resolution.py by scripts/build_notebooks.py. Do not edit here.
"""Resolution extraction schema (SPEC 5.1). Also the fine-tuning target format."""

from enum import StrEnum

from pydantic import BaseModel, Field


class ResolutionType(StrEnum):
    ADOPT_FINANCIALS = "ADOPT_FINANCIALS"
    DIVIDEND = "DIVIDEND"
    DIRECTOR_REAPPOINT_ROTATION = "DIRECTOR_REAPPOINT_ROTATION"
    INDEPENDENT_DIRECTOR_APPOINT = "INDEPENDENT_DIRECTOR_APPOINT"
    NON_INDEPENDENT_DIRECTOR_APPOINT = "NON_INDEPENDENT_DIRECTOR_APPOINT"
    MANAGERIAL_REMUNERATION = "MANAGERIAL_REMUNERATION"
    RELATED_PARTY_TRANSACTION = "RELATED_PARTY_TRANSACTION"
    AUDITOR_APPOINT = "AUDITOR_APPOINT"
    AUDITOR_REMUNERATION_COST = "AUDITOR_REMUNERATION_COST"  # cost auditor ratification
    ESOP = "ESOP"
    CAPITAL_RAISE = "CAPITAL_RAISE"  # QIP, preferential, NCDs
    BORROWING_LIMITS = "BORROWING_LIMITS"  # Sec 180
    ARTICLES_OR_MOA_AMENDMENT = "ARTICLES_OR_MOA_AMENDMENT"
    OTHER = "OTHER"


class Person(BaseModel):
    name: str
    role: str | None = None
    age: int | None = None
    din: str | None = None
    is_promoter: bool | None = None
    tenure_years_proposed: float | None = None
    prior_tenure_years: float | None = None
    board_attendance_pct: float | None = None


class Money(BaseModel):
    amount_inr: float | None = None  # normalized to INR (crore/lakh -> INR)
    raw: str


class ResolutionExtraction(BaseModel):
    item_no: int
    title: str
    resolution_type: ResolutionType
    is_special_resolution: bool
    persons: list[Person] = Field(default_factory=list)
    amounts: list[Money] = Field(default_factory=list)
    counterparty: str | None = None
    counterparty_is_related: bool | None = None
    transaction_nature: str | None = None
    duration_years: float | None = None
    key_facts: list[str] = Field(default_factory=list, max_length=5)

## 3. Model + LoRA

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    BASE_MODEL, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True, dtype=None
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

In [ ]:
def to_text(batch):
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False) for m in batch["messages"]]}

train_ds = ds["train"].map(to_text, batched=True, remove_columns=["messages"])
val_ds = ds["val"].map(to_text, batched=True, remove_columns=["messages"])
lengths = [len(tokenizer(t).input_ids) for t in train_ds["text"]]
print("tokens per example: max", max(lengths), "mean", sum(lengths) // len(lengths))
assert max(lengths) <= MAX_SEQ_LEN, "an example would be truncated"

## 4. Per-epoch extraction metrics (JSON validity, type accuracy)

In [ ]:
def generate(model, messages, max_new_tokens=MAX_NEW_TOKENS):
    """Greedy decode for one chat (system + user); returns the reply and its latency."""
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    started = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    latency_ms = int((time.perf_counter() - started) * 1000)
    return tokenizer.decode(new_tokens, skip_special_tokens=True), latency_ms, len(new_tokens)

In [ ]:
from transformers import TrainerCallback

def score(model, rows):
    """JSON validity before any repair, and resolution_type accuracy vs the teacher."""
    FastLanguageModel.for_inference(model)
    valid = type_ok = 0
    for row in rows:
        text, _, _ = generate(model, row["messages"][:2])
        target = json.loads(row["messages"][2]["content"])
        try:
            pred = ResolutionExtraction.model_validate_json(text)
        except Exception:
            continue
        valid += 1
        type_ok += pred.resolution_type.value == target["resolution_type"]
    FastLanguageModel.for_training(model)
    return {"json_valid": valid / len(rows), "type_acc": type_ok / len(rows), "n": len(rows)}

class ExtractionEvalCallback(TrainerCallback):
    def __init__(self, rows):
        self.rows, self.history = rows, []

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        metrics = {"epoch": round(state.epoch, 2), **score(model, self.rows)}
        self.history.append(metrics)
        print("val subset:", metrics)

eval_rows = ds["val"].shuffle(seed=SEED).select(range(min(EVAL_SUBSET, len(ds["val"]))))
extraction_cb = ExtractionEvalCallback(eval_rows)

## 5. Trainer (loss on assistant tokens only)

In [ ]:
import inspect
from trl import SFTConfig, SFTTrainer
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

def sft_config(**kwargs):
    """SFTConfig across TRL versions (max_seq_length -> max_length, eval_strategy naming)."""
    params = inspect.signature(SFTConfig).parameters
    if "max_seq_length" not in params and "max_length" in params:
        kwargs["max_length"] = kwargs.pop("max_seq_length")
    if "eval_strategy" not in params:
        kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")
    return SFTConfig(**kwargs)

args = sft_config(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    optim="adamw_8bit",
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=5,
    eval_strategy="epoch",
    per_device_eval_batch_size=BATCH,
    save_strategy="steps",
    save_steps=10,
    save_total_limit=3,
    seed=SEED,
    report_to="none",
)
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=args,
    callbacks=[extraction_cb],
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

In [ ]:
# Sanity check: only the assistant JSON should carry loss.
labels = [t for t in trainer.train_dataset[0]["labels"] if t != -100]
print(tokenizer.decode(labels)[:300])

## 6. Train (resumes from the latest Drive checkpoint)

In [ ]:
checkpoints = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
resume = checkpoints[-1] if checkpoints else None
print("resuming from", resume)
train_stats = trainer.train(resume_from_checkpoint=resume)
print(train_stats)

## 7. Metrics (full validation only if FULL_VAL_EVAL)

In [ ]:
final = score(model, ds["val"]) if FULL_VAL_EVAL else None
metrics = {
    "base_model": BASE_MODEL,
    "train_examples": len(ds["train"]),
    "val_examples": len(ds["val"]),
    "epochs": EPOCHS,
    "final_val": final,
    "per_epoch_val_subset": extraction_cb.history,
    "train_loss_log": [h for h in trainer.state.log_history if "loss" in h],
    "eval_loss_log": [h for h in trainer.state.log_history if "eval_loss" in h],
}
with open(f"{OUTPUT_DIR}/metrics.json", "w") as fh:
    json.dump(metrics, fh, indent=2)
print(json.dumps(final, indent=2))

## 8. Save the LoRA adapter (private HF repo)

In [ ]:
model.save_pretrained(f"{OUTPUT_DIR}/final")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final")
model.push_to_hub(LORA_REPO, token=HF_TOKEN, private=True)
tokenizer.push_to_hub(LORA_REPO, token=HF_TOKEN, private=True)
HfApi(token=HF_TOKEN).upload_file(path_or_fileobj=f"{OUTPUT_DIR}/metrics.json",
                                  path_in_repo="metrics.json", repo_id=LORA_REPO)

## 9. Export a merged Q4_K_M GGUF for CPU serving (llama.cpp)

In [ ]:
model.save_pretrained_gguf("gguf_out", tokenizer, quantization_method="q4_k_m")
candidates = [p for p in glob.glob("**/*.gguf", recursive=True) if "q4_k_m" in p.lower()]
assert candidates, "GGUF export produced no Q4_K_M file"
gguf_path = max(candidates, key=os.path.getmtime)
print(gguf_path, round(os.path.getsize(gguf_path) / 1e9, 2), "GB")

api = HfApi(token=HF_TOKEN)
api.create_repo(GGUF_REPO, private=True, exist_ok=True)
api.upload_file(path_or_fileobj=gguf_path, path_in_repo=GGUF_FILE, repo_id=GGUF_REPO)
print(f"Uploaded https://huggingface.co/{GGUF_REPO}/blob/main/{GGUF_FILE}")

## Next

In the repo's `.env` set `STUDENT_GGUF_REPO=<you>/proxylens-qwen3b-gguf-v1` (and `HF_TOKEN`), then run
the Phase 4 smoke test:

```bash
uv sync --extra student
uv run python -m scripts.smoke_student --n 10
```